In [ ]:
from paddleocr import PPStructure, PaddleOCR
from PIL import Image, ImageDraw

img_path = 'data/a_2_2.jpg'

# --- 1. Первый проход: ищем синюю рамку (анкета) ---
table_engine = PPStructure(show_log=True)
result = table_engine(img_path)

# Найдём синий блок (обычно type == 'table' или largest bbox)
anketa_bbox = None
anketa_bboxes = []
max_area = 0
for region in result:
    bbox = region['bbox']
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    area = w * h
    # Можно вставить проверку, что этот блок подходит вам по координатам
    if area > max_area:
        if abs(area - max_area) < 100:
          anketa_bboxes.append(bbox)
        else:
          max_area = area
          anketa_bbox = bbox

# --- 2. Обрезаем анкету ---
img = Image.open(img_path)
anketa_img = img.crop(anketa_bbox)
anketa_img.save('anketa_crop.jpg')

# --- 3. Второй pass: ищем поля внутри анкеты ---
# Можно использовать снова PPStructure или просто PaddleOCR для детекции всех текстовых блоков
ocr = PaddleOCR(use_angle_cls=True, lang="ru")  # или lang="en"
result_anketa = ocr.ocr('anketa_crop.jpg', cls=True)

# --- 4. Визуализация найденных полей ---
draw = ImageDraw.Draw(anketa_img)
for line in result_anketa[0]:    # result_anketa[0] такое у paddleocr
    box = line[0]
    xy = [tuple(point) for point in box]
    draw.polygon(xy, outline='red', width=2)
    txt = line[1][0]
    draw.text(xy[0], txt, fill='blue')

anketa_img.save('anketa_with_fields.jpg')
anketa_img.show()

# --- 5. Сохраняем каждый текстовый блок как отдельный файл ---
for idx, line in enumerate(result_anketa[0]):
    box = line[0]
    xmin = int(min(point[0] for point in box))
    ymin = int(min(point[1] for point in box))
    xmax = int(max(point[0] for point in box))
    ymax = int(max(point[1] for point in box))
    field_crop = anketa_img.crop((xmin, ymin, xmax, ymax))
    field_crop.save(f'new_data/anketa_field_{idx}.jpg')

[2025/04/29 14:49:05] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, use_gcu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='/root/.paddleocr/whl/det/ch/ch_PP-OCRv4_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='/root/.paddleocr/whl/rec/ch/ch_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_l

In [ ]:
from paddleocr import PPStructure, PaddleOCR
from PIL import Image, ImageDraw
import torch
from torchvision import transforms
import os

# Пути к файлам
img_path = 'data/a_1_1.jpg'
output_folder = 'outputs'
os.makedirs(output_folder, exist_ok=True)

# --- 1. Первый проход: ищем синюю рамку (анкета) ---
table_engine = PPStructure(show_log=True)
result = table_engine(img_path)

# Найдём самый крупный блок (предположительно, это анкета)
anketa_bbox = None
max_area = 0
for region in result:
    bbox = region['bbox']
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    area = w * h
    if area > max_area:
        max_area = area
        anketa_bbox = bbox

# --- 2. Обрезаем анкету ---
img = Image.open(img_path)
anketa_img = img.crop(anketa_bbox)
anketa_img.save('anketa_crop.jpg')

# --- 3. Второй проход: ищем текстовые блоки внутри анкеты ---
ocr = PaddleOCR(use_angle_cls=True, lang="ru")
result_anketa = ocr.ocr('anketa_crop.jpg', cls=True)

# --- 4. Визуализация найденных полей ---
draw = ImageDraw.Draw(anketa_img)
for line in result_anketa[0]:
    box = line[0]
    xy = [tuple(point) for point in box]
    draw.polygon(xy, outline='red', width=2)
    txt = line[1][0]
    draw.text(xy[0], txt, fill='blue')
anketa_img.save('anketa_with_fields.jpg')
anketa_img.show()

# --- 5. Классификатор рукописного текста ---
# Загрузка модели
model = torch.load('handwriting_classifier.pt', map_location=torch.device('cpu'))
model.eval()

# Преобразования изображения для модели
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Функция: определение рукописного текста
def is_handwritten(img_path):
    image = Image.open(img_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0)
    with torch.no_grad():
        output = model(input_tensor)
        predicted = torch.argmax(output, dim=1)
        return predicted.item() == 1  # 1 = рукописный

# --- 6. Обработка каждого блока ---
for idx, line in enumerate(result_anketa[0]):
    box = line[0]
    xmin = int(min(point[0] for point in box))
    ymin = int(min(point[1] for point in box))
    xmax = int(max(point[0] for point in box))
    ymax = int(max(point[1] for point in box))

    field_crop = anketa_img.crop((xmin, ymin, xmax, ymax))
    field_path = os.path.join(output_folder, f'anketa_field_{idx}.jpg')
    field_crop.save(field_path)

    # Классификация: рукописный или нет
    if is_handwritten(field_path):
        # Применяем OCR
        result = ocr.ocr(field_path, cls=True)
        print(f"[HANDWRITTEN OCR] Field {idx}: {result}")
    else:
        print(f"[SKIPPED] Field {idx} — not handwritten")


[2025/04/29 15:02:49] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, use_gcu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='/root/.paddleocr/whl/det/ch/ch_PP-OCRv4_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='/root/.paddleocr/whl/rec/ch/ch_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_l

In [8]:
!pip install paddlepaddle
!pip install paddleocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: opt_einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0


In [18]:
from paddleocr import PPStructure, PaddleOCR
from PIL import Image, ImageDraw

img_path = '/content/a_2_1.jpg'

# --- 1. Первый проход: ищем все таблицы ---
table_engine = PPStructure(show_log=True)
result = table_engine(img_path)

# Собираем все bbox таблиц
table_bboxes = [region['bbox'] for region in result if region['type'] == 'table']

# Если 2 таблицы — возможно, нужно их объединить
def area(bbox):
    return (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])

def vertical_overlap(b1, b2, threshold=0.1):
    # Проверяет, расположены ли таблицы вертикально
    h1_center = (b1[1] + b1[3]) / 2
    h2_center = (b2[1] + b2[3]) / 2
    return abs(h1_center - h2_center) > (b1[3] - b1[1]) * threshold

anketa_bbox = None

if len(table_bboxes) == 2:
    a1 = area(table_bboxes[0])
    a2 = area(table_bboxes[1])
    if abs(a1 - a2) / max(a1, a2) < 0.2 and vertical_overlap(table_bboxes[0], table_bboxes[1]):
        # Объединяем два bbox
        x0 = min(table_bboxes[0][0], table_bboxes[1][0])
        y0 = min(table_bboxes[0][1], table_bboxes[1][1])
        x1 = max(table_bboxes[0][2], table_bboxes[1][2])
        y1 = max(table_bboxes[0][3], table_bboxes[1][3])
        anketa_bbox = [x0, y0, x1, y1]
    else:
        # Берем самую большую таблицу
        anketa_bbox = max(table_bboxes, key=area)
else:
    # Если одна таблица или больше двух — берём самую большую
    anketa_bbox = max(table_bboxes, key=area) if table_bboxes else None

# --- 2. Обрезаем анкету ---
img = Image.open(img_path)
anketa_img = img.crop(anketa_bbox)
anketa_img.save('anketa_crop.jpg')


[2025/04/29 15:42:00] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, use_gcu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='/root/.paddleocr/whl/det/ch/ch_PP-OCRv4_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='/root/.paddleocr/whl/rec/ch/ch_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_l

In [22]:
# --- 2. Визуализация bbox и обрезка анкеты ---
img = Image.open(img_path)

# Визуализируем объединённый bbox
visual_img = img.copy()
draw = ImageDraw.Draw(visual_img)
draw.rectangle(anketa_bbox, outline='green', width=4)
visual_img.save('anketa_bbox_debug.jpg')
visual_img.show()

# Обрезаем изображение по bbox
anketa_img = img.crop(anketa_bbox)
anketa_img.save('anketa_crop.jpg')


In [20]:
from paddleocr import PPStructure, PaddleOCR
from PIL import Image

def get_best_anketa_bbox(img_path, expand_margin=20):
    # Загружаем изображение
    img = Image.open(img_path)
    width, height = img.size

    # --- 1. PPStructure (ищем таблицы) ---
    structure_engine = PPStructure(show_log=False)
    structure_result = structure_engine(img_path)

    table_bboxes = [r['bbox'] for r in structure_result if r['type'] == 'table']

    def area(bbox):
        return (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])

    def vertical_overlap(b1, b2):
        # Проверка, что таблицы расположены одна под другой
        return abs(((b1[1]+b1[3])/2) - ((b2[1]+b2[3])/2)) > (b1[3]-b1[1]) * 0.2

    # --- Обработка таблиц ---
    if len(table_bboxes) >= 2:
        # Проверим, есть ли похожие по размеру — объединяем
        a1, a2 = area(table_bboxes[0]), area(table_bboxes[1])
        if abs(a1 - a2) / max(a1, a2) < 0.2 and vertical_overlap(table_bboxes[0], table_bboxes[1]):
            boxes = table_bboxes[:2]
        else:
            boxes = table_bboxes
    elif len(table_bboxes) == 1:
        # Попробуем немного расширить bbox
        box = table_bboxes[0]
        x0 = max(0, box[0] - expand_margin)
        y0 = max(0, box[1] - expand_margin)
        x1 = min(width, box[2] + expand_margin)
        y1 = min(height, box[3] + expand_margin)
        return [x0, y0, x1, y1]
    else:
        boxes = []

    if boxes:
        x0 = min(b[0] for b in boxes)
        y0 = min(b[1] for b in boxes)
        x1 = max(b[2] for b in boxes)
        y1 = max(b[3] for b in boxes)
        return [x0, y0, x1, y1]

    # --- 2. Fallback: OCR — объединить все текстовые блоки ---
    ocr = PaddleOCR(use_angle_cls=True, lang='ru')
    ocr_result = ocr.ocr(img_path, cls=True)
    text_boxes = [line[0] for line in ocr_result[0]]

    if not text_boxes:
        return None

    x0 = min(pt[0][0] for box in text_boxes for pt in box)
    y0 = min(pt[0][1] for box in text_boxes for pt in box)
    x1 = max(pt[2][0] for box in text_boxes for pt in box)
    y1 = max(pt[2][1] for box in text_boxes for pt in box)
    return [int(x0), int(y0), int(x1), int(y1)]


In [21]:
img_path = '/content/a_2_1.jpg'
anketa_bbox = get_best_anketa_bbox(img_path)

if anketa_bbox:
    img = Image.open(img_path)
    anketa_img = img.crop(anketa_bbox)
    anketa_img.save('anketa_crop.jpg')
else:
    print("Не удалось найти подходящий bbox")